# Reading from the silver table/s

In [0]:
df_prd = spark.read.table("workspace.silver.crm_prd_info")
df_cat = spark.read.table("workspace.silver.erp_px_cat_g1v2")


# Creating temporary view

In [0]:
# Register them as Temporary Views so Spark SQL can see them
df_prd.createOrReplaceTempView("pi")
df_cat.createOrReplaceTempView("pc")


# Create customers dimension table

In [0]:
query = """
    SELECT
        ROW_NUMBER() over(order by pi.sales_product_key, pi.product_start_date) as product_surrogate_key,
        pi.product_id,
        pi.sales_product_key,
        pi.product_name,
        pi.category_id,
        pc.category,
        pc.sub_category,
        pc.maintenance,
        pi.product_cost,
        pi.product_line,
        pi.product_start_date
    FROM pi
    LEFT JOIN pc
    ON pi.category_id = pc.id
    WHERE pi.product_end_date IS NULL --filter out all the historical data


"""

df = spark.sql(query)


# Write to Gold

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("gold.dim_products")
)

In [0]:
%sql
select * from workspace.gold.dim_products